<a href="https://colab.research.google.com/github/DibyaSadhukhan/AI_Playround/blob/main/Career%20Assistant%20Master%20-%20Cover%20Letter%20Generator%20and%20Email%20Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,292 kB]


In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [3]:
!ollama pull llama3.2
!ollama pull nomic-embed-text

In [4]:
!pip install langchain_core
!pip install langchain-ollama
!pip install langchain_community
!pip install langchain_text_splitters
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [5]:
import subprocess
import json
import tempfile
import requests

URL = "https://raw.githubusercontent.com/DibyaSadhukhan/portfolio_website/refs/heads/main/src/data/content.ts"

# 1. Fetch TS file
ts_code = requests.get(URL).text

# 2. Wrap it so Node can evaluate it
# - remove 'export'
# - return all variables as one object
wrapped_js = f"""
{ts_code.replace("export const", "const")}

console.log(JSON.stringify({{
    portfolioData,
    icons,
    core_skills,
    capsule_skills,
    experience_data,
    projects_data
}}));
"""

# 3. Write temp JS file
with tempfile.NamedTemporaryFile(suffix=".js", delete=False, mode="w") as f:
    f.write(wrapped_js)
    temp_path = f.name

# 4. Run Node
result = subprocess.run(
    ["node", temp_path],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("❌ Node error:\n", result.stderr)
    raise Exception("JS execution failed")

# 5. Load into Python dict
data = json.loads(result.stdout)

# print("✅ Loaded:", type(data))
# print(data.keys())

✅ Loaded: <class 'dict'>
dict_keys(['portfolioData', 'icons', 'core_skills', 'capsule_skills', 'experience_data', 'projects_data'])


In [7]:
from langchain_core.documents import Document

docs = []

for project in data["projects_data"]:

    text = f"""
    Project: {project["title"]}
    Category: {project["category"]}
    Description: {project["description"]}
    Tools: {", ".join(project["tools"])}
    """

    docs.append(Document(page_content=text))

for experience in data["experience_data"]:

    text = f"""
    role: {experience["role"]}
    company: {experience["company"]}
    Description: {experience["description"]}
    Tools: {", ".join(experience["tools"])}
    """

    docs.append(Document(page_content=text))
profile_data =  f"""
    core_details: {data["portfolioData"]} """
docs.append(Document(page_content=profile_data))
# print(docs)

[Document(metadata={}, page_content='\n    Project: AI Blog Evaluation & Feedback System\n    Category: GenAI / Automation\n    Description: An LLM-powered content evaluation system that analyzes blog submissions, provides paragraph-level feedback, and ensures consistency with organizational writing style through real-time tooltips and automated email reports.\n    Tools: PHP, JavaScript, Fine-tuned ChatGPT, PostgreSQL (Vector DB), n8n, Prompt Engineering\n    '), Document(metadata={}, page_content='\n    Project: AI-Powered Suicide Prevention Helpline\n    Category: GenAI / Voice AI / Automation\n    Description: An AI-driven voice-based support system that engages users in distress, performs real-time de-escalation, and intelligently routes high-risk cases to human agents.\n    Tools: Node.js, Python, Custom LLMs, Speech-to-Text, Text-to-Speech, MLOps, APIs\n    '), Document(metadata={}, page_content='\n    Project: Heart It Out - Data Infrastructure, Automation and Backend Systems\n

In [8]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

chunked_docs = splitter.split_documents(docs)

# print(len(chunked_docs))
# print(chunked_docs[0].page_content)

38
Project: AI Blog Evaluation & Feedback System
    Category: GenAI / Automation
    Description: An LLM-powered content evaluation system that analyzes blog submissions, provides paragraph-level feedback, and ensures consistency with organizational writing style through real-time tooltips and automated email reports.


In [9]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [11]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunked_docs, embeddings)

In [12]:
query = "What projects use n8n?"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

Tools: PHP, JavaScript, n8n, SQL, System Design, Automation, APIs, Cloud Hosting
------
Project: n8n Automated Backup & Restore System
    Category: Automation / DevOps / Reliability
    Description: Built an automated backup and versioning system for n8n workflows to prevent data loss, ensuring reliable recovery from database failures.
    Tools: n8n, Node.js, AWS S3, Google Drive API, APIs, Automation, System Design
------
Project: Microservices-Based eCommerce Platform (n8n Deployable)
    Category: Automation / Microservices / Backend Systems
    Description: Built a fully modular eCommerce platform powered by n8n workflows, enabling one-click deployment and server-agnostic scalability through microservices architecture.
    Tools: n8n, Node.js, APIs, Microservices, JavaScript, SQL, System Design, Cloud
------


In [13]:
from langchain_ollama.llms import OllamaLLM

llm = OllamaLLM(model="llama3.2")

In [14]:
"""
 query = "My project needs automation of my business so that it is tech indepent can it be done?"

 results = vectorstore.similarity_search(query, k=3)

 context = "\n\n".join([r.page_content for r in results])

 prompt = f
You are an assistant answering questions about my professional experience to potential clients

Rules:
- The answers should be in third person example : "Dibya's Projects , His Skills , He has demostrated"
- Only use the provided context
- If the answer is not clearly in the context, say "I don't have enough information"
- Be elaborate and boast about the projects and my skills like you are trying to land me this client

Context:
{context}

Question:
{query}

Answer:


response = llm.invoke(prompt)

print(response)
"""

Dibya's extensive experience in designing and developing end-to-end automation systems has equipped him with the expertise to transform your business into a tech-independent entity. By leveraging his skills in building intelligent automation platforms, deploying NLP models, and integrating AI solutions, he can help you streamline core business processes, improving operational efficiency and reducing manual effort.

Dibya has successfully implemented automation workflows that have automated lead-to-meeting conversion, scheduling systems integrated with Google Calendar APIs, and email automation using Google Email APIs. These projects demonstrate his ability to develop scalable system architecture from the ground up, design and implement backend services using PHP and JavaScript frameworks, and integrate AI solutions into production workflows.

With Dibya's expertise in cloud-based data systems, Python, ML frameworks, and NLP models, he can help you create a tech-independent business tha

In [18]:
memory = []
MAX_MEMORY = 6   # keep last 3 interactions (prompt+response pairs)
def generate_response(query: str, instruction: str) -> str:
    # Retrieve career context (RAG)
    results = vectorstore.similarity_search(query, k=4)
    context = "\n\n".join([r.page_content for r in results])

    # Build memory context (last few interactions only)
    memory_context = "\n".join([
        f"Prompt: {m['prompt']}\nResponse: {m['response']}"
        for m in memory
    ])

    prompt = f"""
You are an AI assistant helping generate job application content.

Context (career data):
{context}

Previous interactions (for consistency, do not copy blindly):
{memory_context}

Instruction:
{instruction}

Rules:
- Stay consistent with previous responses
- Do not repeat the same sentences
- Keep outputs natural and professional

Output:
"""

    response = llm.invoke(prompt)

    # Store current interaction
    memory.append({
        "prompt": instruction,
        "response": response
    })

    # Trim memory (avoid explosion)
    if len(memory) > MAX_MEMORY:
        memory.pop(0)

    return response

In [ ]:
# Paste JD here

cover_letter = generate_response(
    query="Summarize my experience, projects, and skills",
    instruction="""
Write a professional cover letter.
For this job description :

Responsibilities:

Design, implement, and deploy generative AI models (GPT, VAEs, GANs) to solve complex problems. Optimize models for performance, scalability, and efficiency. Build RAG (Retrieval-Augmented Generation) pipelines and integrate vector databases (Chroma DB, PineCone, FAISS). Conduct research to apply the latest breakthroughs in NLP and computer vision. Collaborate with data scientists and engineers to integrate AI capabilities into existing products.

y

Mandatory skill sets:

LangChain, Langgraph, RAG,L Lama

Preferred skill sets:

LangChain, Langgraph, RAG,L Lama

Years of experience required:

2 to 10 years

Education qualification:

Graduate Engineer or Management Graduate

Education (if blank, degree and/or field of study not specified)

Degrees/Field of Study required: Master of Business Administration, Bachelor of Engineering
Degrees/Field of Study preferred:
Certifications (if blank, certifications not specified)

Required Skills

Data Engineering
Optional Skills

Accepting Feedback, Accepting Feedback, Active Listening, Agile Scalability, Amazon Web Services (AWS), Analytical Thinking, Apache Airflow, Apache Hadoop, Azure Data Factory, Communication, Creativity, Data Anonymization, Data Architecture, Database Administration, Database Management System (DBMS), Database Optimization, Database Security Best Practices, Databricks Unified Data Analytics Platform, Data Engineering, Data Engineering Platforms, Data Infrastructure, Data Integration, Data Lake, Data Modeling, Data Pipeline {+ 28 more}
Desired Languages (If blank, desired languages not specified)

Travel Requirements

Not Specified
Available for Work Visa Sponsorship?

No
Government Clearance Required?

No


Requirements:
- Tailored for AI Engineer / Data Engineer roles
- Highlight experience with LLMs, automation, and backend systems
- Keep it concise (200–300 words)
- Use a confident but not arrogant tone
"""
)

print(cover_letter)

In [ ]:
recruiter_email = generate_response(
    query="What are my strongest projects and skills?",
    instruction="""
Write a cold outreach email to a recruiter.

Requirements:
- Short (100–150 words)
- Mention key strengths and relevant experience
- Include interest in AI/Data roles
- Keep it direct and professional
"""
)

print(recruiter_email)

In [ ]:
followup_email = generate_response(
    query="What are my key achievements and strengths?",
    instruction="""
Write a polite follow-up email after no response.

Requirements:
- Very concise (60–100 words)
- Reiterate interest
- Mention 1–2 strong points briefly
- Professional and respectful tone
"""
)

print(followup_email)